# Benchmark
A comprehensive comparison against state-of-the-art causal discovery methodologies is performed. <br>
Each selected method exemplifies a distinct category within the spectrum of causal inference approaches. <br>
Namely, we includes the Pairwise Granger Causality test implementation available in the `statsmodels` package for Python <br>

From the constraint-based family, we select the PCMCI algorithm developed by Runge, which is implemented in the Tigramite Python package. <br> 
The VarLiNGAM method, as proposed by Hyvärinen et al., is our choice for the noise-based category.  <br> 
It is implemented in Python via the LiNGAM library .   <br> 

Finally, for the score-based category, we incorporate DYNOTEARS, a method introduced by Pamfil et al. and implemented in the CausalNex Python library.  <br> 

We also include standard VAR modeling, adopting the regression coefficients as if they were causal.  <br> 


It is important to note that each method has its own underlying assumptions, which might not always be respected in practical scenarios. For example, VarLiNGAM assumes non-Gaussian errors, which is not the case in our experiments. Similarly, we use PCMCI with the ParCorr independence test; while a nonlinear test would be more appropriate. We faced computational challenges with PCMCI when attempting to use its nonlinear inpendence test CMIknn. Nevertheless, our goal is not to demonstrate that our approach outperforms all others under all conditions. Instead, we aim to show that our method can be a valuable addition to the toolbox for causal discovery in time series, offering unique insights and potentially complementing existing techniques. The adopted conditions for each method might be suboptimal for the given dataset, yet they provide a robust benchmark to evaluate the relative strengths and potential applications of our proposed approach.

Each method has been wrapped conveniently to uniformize the way to create the objects, run the execution, and return the same structure.  

In [9]:
import os

# This is because VARLINGAM will use all available CPU with n_jobs > 1 - Limit to 1 thread
os.environ['MKL_NUM_THREADS'] = '1'  
os.environ['NUMEXPR_NUM_THREADS'] = '1'
os.environ['OMP_NUM_THREADS'] = '1'
os.environ['OPENBLAS_NUM_THREADS'] = '1'
os.environ['OMP_NUM_THREADS'] = '1'

import sys
sys.path.append('../src')


import pickle 
import os
from d2c.descriptors import DataLoader
from d2c.benchmark import VARLiNGAM, PCMCI, Granger, DYNOTEARS, D2CWrapper, VAR, MultivariateGranger

from imblearn.ensemble import BalancedRandomForestClassifier

#suppress future warning
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)


In [10]:
N_JOBS = 50
MAXLAGS = 3
N_VARS = 5

In [11]:
dataloader = DataLoader(n_variables = N_VARS,
                            maxlags = MAXLAGS)
dataloader.from_pickle(f'realdata/netsym/netsym_5.pkl')

original_observations_testing= dataloader.get_original_observations()
lagged_flattened_observations_testing = dataloader.get_observations()
flattened_dags_testing= dataloader.get_dags()
true_causal_dfs = dataloader.get_true_causal_dfs()

Each method from the benchmark will take as input 
- `ts_list`: a list of `np.arrays` containing the values of the time series, 
- `maxlags`: the maxlags, 
- `n_jobs`: the number of jobs. <br>
It's important to notice that `get_causal_dfs()` will return a dictionary of dataframes where the key is the index of the corresponding time series from the input list `ts_list`.
So, if our data contains `15` time series and you want to access the last one we can do `causal_dfs[15 - 1]`

## Competitors

In [18]:
var = VAR(ts_list=original_observations_testing, maxlags=MAXLAGS, n_jobs=N_JOBS)
var.run()
causal_dfs_var = var.get_causal_dfs()

Running parallel inference over 1050 time series using 50 jobs...


Processing Time Series:   0%|          | 0/1050 [00:00<?, ?it/s]

In [19]:
varlingam = VARLiNGAM(ts_list=original_observations_testing, maxlags=MAXLAGS, n_jobs=N_JOBS)
varlingam.run()
causal_dfs_varlingam = varlingam.get_causal_dfs()

Running parallel inference over 1050 time series using 50 jobs...


Processing Time Series:   0%|          | 0/1050 [00:00<?, ?it/s]

In [20]:
pcmci = PCMCI(ts_list=original_observations_testing, maxlags=MAXLAGS, n_jobs=N_JOBS)
pcmci.run()
causal_dfs_pcmci = pcmci.get_causal_dfs()

Running parallel inference over 1050 time series using 50 jobs...


Processing Time Series:   0%|          | 0/1050 [00:00<?, ?it/s]

In [16]:
pcmci_gdpc = PCMCI(ts_list=original_observations_testing, maxlags=MAXLAGS, n_jobs=N_JOBS, ci ='GPDC')
pcmci_gdpc.run()
causal_dfs_pcmci_gdpc = pcmci_gdpc.get_causal_dfs()

Running parallel inference over 1050 time series using 50 jobs...


Processing Time Series:   0%|          | 0/1050 [00:00<?, ?it/s]

/home/gpaldino/miniconda3/envs/td2c/lib/python3.8/site-packages/sklearn/gaussian_process/kernels.py:429: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__length_scale is close to the specified upper bound 100000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/home/gpaldino/miniconda3/envs/td2c/lib/python3.8/site-packages/sklearn/gaussian_process/kernels.py:419: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__length_scale is close to the specified lower bound 1e-05. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(
/home/gpaldino/miniconda3/envs/td2c/lib/python3.8/site-packages/sklearn/gaussian_process/kernels.py:419: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__length_scale is close to the specified lower bound 1e-05. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(
/home/gpaldino/minicon

In [22]:
granger = Granger(ts_list=original_observations_testing, maxlags=MAXLAGS, n_jobs=N_JOBS)
granger.run()
causal_dfs_granger = granger.get_causal_dfs()

Running parallel inference over 1050 time series using 50 jobs...


Processing Time Series:   0%|          | 0/1050 [00:00<?, ?it/s]

In [23]:
dynotears = DYNOTEARS(ts_list=original_observations_testing, maxlags=MAXLAGS, n_jobs=N_JOBS)
dynotears.run()
causal_dfs_dynotears = dynotears.get_causal_dfs()

Running parallel inference over 1050 time series using 50 jobs...


Processing Time Series:   0%|          | 0/1050 [00:00<?, ?it/s]

In [24]:
mvgc = MultivariateGranger(ts_list=original_observations_testing, maxlags=MAXLAGS, n_jobs=N_JOBS)
mvgc.run()
causal_dfs_mvgc = mvgc.get_causal_dfs()

Running parallel inference over 1050 time series using 50 jobs...


Processing Time Series:   0%|          | 0/1050 [00:00<?, ?it/s]

In [3]:
# save all in a temp folder pickle 
temp_folder = 'data/benchmark_results'
if not os.path.exists(temp_folder):
    os.makedirs(temp_folder)


In [19]:
causal_dfs_pcmci_gdpc

{49:     from  to    effect p_value probability is_causal
 0      5   0  0.381039     0.0        None         1
 1      5   1  0.098962   0.602        None         0
 2      5   2  0.112915   0.402        None         0
 3      5   3  0.176595   0.026        None         1
 4      5   4  0.155604   0.052        None         0
 ..   ...  ..       ...     ...         ...       ...
 70    19   0  0.080219   0.912        None         0
 71    19   1  0.101055   0.564        None         0
 72    19   2  0.135885    0.16        None         0
 73    19   3  0.081812   0.886        None         0
 74    19   4  0.082832   0.874        None         0
 
 [75 rows x 6 columns],
 45:     from  to    effect p_value probability is_causal
 0      5   0  0.247724     0.0        None         1
 1      5   1  0.104154    0.51        None         0
 2      5   2  0.089685    0.76        None         0
 3      5   3  0.174355   0.026        None         1
 4      5   4  0.107287   0.464        None     

In [17]:

all = {
    'causal_dfs_var': causal_dfs_var,
    'causal_dfs_varlingam': causal_dfs_varlingam,
    'causal_dfs_pcmci': causal_dfs_pcmci,
    'causal_dfs_pcmci_gpdc': causal_dfs_pcmci_gdpc, 
    'causal_dfs_granger': causal_dfs_granger,
    'causal_dfs_mvgc': causal_dfs_mvgc,
    'causal_dfs_dynotears': causal_dfs_dynotears,
    'observations': original_observations_testing,
    'dags': flattened_dags_testing,
    'true_causal_dfs': true_causal_dfs
}
pickle.dump(all, open(os.path.join(temp_folder, 'causal_dfs_before_d2c_netsim_5.pkl'), 'wb'))

In [4]:
# load everything
all = pickle.load(open(os.path.join(temp_folder, 'causal_dfs_before_d2c_netsim_5.pkl'), 'rb'))
causal_dfs_var = all['causal_dfs_var']
causal_dfs_varlingam = all['causal_dfs_varlingam']
causal_dfs_pcmci = all['causal_dfs_pcmci']
causal_dfs_pcmci_gpdc = all['causal_dfs_pcmci_gpdc']
causal_dfs_granger = all['causal_dfs_granger']
causal_dfs_mvgc = all['causal_dfs_mvgc']
causal_dfs_dynotears = all['causal_dfs_dynotears']

## D2CWrapper
For coherence with the other results, a D2CWrapper class has been created that behave exactly like the other approaches.<br>
It therefore exposes the methods `run()` and `get_causal_dfs()`. <br>
It requires a model that has been trained already and it will compute descriptors for unseen data of which the DAG is ignored. <br>
In this case, the model cannot select a subset of features (no `couples_to_consider_per_dag` attribute).
The predictions from the model on the newly computed descriptors are the labels that will be provided in the causal df. <br>

<b>Important:</b> make sure your model has been trained on the same feature set. If you have used `full=True` when generating the training descriptors, you should use `full=True` here as well

In [5]:
import pandas as pd
descriptors_df_train = pd.read_pickle('data/descriptors_df_train_raw.pkl')
descriptors_df_train.fillna(descriptors_df_train.mean(), inplace=True)

X_train = descriptors_df_train.drop(columns=['graph_id','edge_source','edge_dest','is_causal'])
y_train = descriptors_df_train['is_causal']

clf = BalancedRandomForestClassifier(n_estimators=500, max_depth=None, random_state=0, sampling_strategy='auto',replacement=True,bootstrap=True)
clf.fit(X_train, y_train)

BalancedRandomForestClassifier(bootstrap=True, n_estimators=500, random_state=0,
                               replacement=True, sampling_strategy='auto')

In [6]:
print(X_train.columns)

Index(['copula_loglik', 'copula_tau_0', 'copula_tau_1', 'copula_tau_2',
       'copula_tau_3', 'copula_tau_4', 'copula_tau_5', 'parcorr_errors',
       'errors_correlation_with_inputs', 'coeff_cause', 'coeff_eff', 'HOC_3_1',
       'HOC_1_2', 'HOC_2_1', 'HOC_1_3', 'kurtosis_ca', 'kurtosis_ef',
       'te_asymmetry_diff_1_15', 'transfer_entropy_fwd',
       'transfer_entropy_bwd', 'transfer_entropy_diff', 'com_cau', 'cau_eff',
       'eff_cau', 'eff_cau_mbeff', 'cau_eff_mbcau', 'skewness_ca',
       'skewness_ef', 'mca_mef_cau_parent', 'mca_mef_cau_child',
       'mca_mef_eff_parent', 'mca_mef_eff_child', 'cau_m_eff_interaction',
       'eff_m_cau_parent', 'eff_m_cau_child', 'm_cau_interaction',
       'eff_cau_mbcau_plus_interaction', 'cau_eff_mbeff_plus_parent',
       'cau_eff_mbeff_plus_child', 'm_eff_parent', 'm_eff_child',
       'mca_mca_cau_parent', 'mca_mca_cau_child', 'mbe_mbe_eff_interaction',
       'mca_mef_cau_interaction', 'mca_mef_eff_interaction',
       'eff_m_cau_inte

In [13]:
d2cwrapper = D2CWrapper(
    ts_list=original_observations_testing,
    model=clf,
    n_variables=N_VARS,
    maxlags=MAXLAGS,
    mb_estimator = 'ts',
    n_jobs=N_JOBS, 
    full=True,
    dynamic=True,
    manages_own_parallelism=False,
)


d2cwrapper.run()
causal_dfs_d2c = d2cwrapper.get_causal_dfs()

Running parallel inference over 1050 time series using 50 jobs...


Processing Time Series:   0%|          | 0/1050 [00:00<?, ?it/s]

## Saving

In [21]:
causal_dfs_pcmci_gdpc

{49:     from  to    effect p_value probability is_causal
 0      5   0  0.381039     0.0        None         1
 1      5   1  0.098962   0.602        None         0
 2      5   2  0.112915   0.402        None         0
 3      5   3  0.176595   0.026        None         1
 4      5   4  0.155604   0.052        None         0
 ..   ...  ..       ...     ...         ...       ...
 70    19   0  0.080219   0.912        None         0
 71    19   1  0.101055   0.564        None         0
 72    19   2  0.135885    0.16        None         0
 73    19   3  0.081812   0.886        None         0
 74    19   4  0.082832   0.874        None         0
 
 [75 rows x 6 columns],
 45:     from  to    effect p_value probability is_causal
 0      5   0  0.247724     0.0        None         1
 1      5   1  0.104154    0.51        None         0
 2      5   2  0.089685    0.76        None         0
 3      5   3  0.174355   0.026        None         1
 4      5   4  0.107287   0.464        None     

In [22]:
# sort every dictionary by key and assess
def sort_dict_by_key(d):
    return {k: d[k] for k in sorted(d)}

# sort the causal dfs by key
causal_dfs_var = sort_dict_by_key(causal_dfs_var)
causal_dfs_varlingam = sort_dict_by_key(causal_dfs_varlingam)
causal_dfs_pcmci = sort_dict_by_key(causal_dfs_pcmci)
causal_dfs_pcmci_gpdc = sort_dict_by_key(causal_dfs_pcmci_gdpc)
causal_dfs_granger = sort_dict_by_key(causal_dfs_granger)
causal_dfs_mvgc = sort_dict_by_key(causal_dfs_mvgc)
causal_dfs_dynotears = sort_dict_by_key(causal_dfs_dynotears)
causal_dfs_d2c = sort_dict_by_key(causal_dfs_d2c)

# assert that the keys are the same
assert set(causal_dfs_var.keys()) == set(causal_dfs_varlingam.keys()) == set(causal_dfs_pcmci.keys()) == set(causal_dfs_granger.keys()) == set(causal_dfs_dynotears.keys()) == set(causal_dfs_d2c.keys())


In [23]:
with open('data/causal_dfs_netsim_5.pkl', 'wb') as f:
    pickle.dump((causal_dfs_var, 
                causal_dfs_varlingam, 
                causal_dfs_pcmci,
                causal_dfs_mvgc,
                causal_dfs_pcmci_gdpc,
                causal_dfs_granger, 
                causal_dfs_dynotears,
                causal_dfs_d2c, 
                true_causal_dfs), f)